# Strict Isomorphism

## **2. (a)**

### **General idea: For $A_1=(Q_1,Σ,τ_1,(q_s)_1,F_1)$ and $A_2=(Q_2,Σ,τ_2,(q_s)_2,F_2)$**

**To be strictly isomorphic**

--Define a function is_strict_isomorphism to test if A1 and A2 are strictly isomorphic

--Need a bijection from $Q_1$ to $Q_2$, so must have same cardinalities ($|Q_1|=|Q_2|$), return False if not

--Need $f(F_1)=F_2$, since $f$ is bijection, must have $|F_1|=|F_2|$, return False if not

Now we can construct the mapping $f$ using backtrack search

**Note:** that we assumed that $Q={0,1,...,n-1}$ for some natural number $n$, with $|Q_1|=|Q_2|$ if $f$ is an injection, it is a bijection

--Create a list to track the used Q2 states, this insures injectivity of f

--Want $f$ to map $(q_s)_1$ to $(q_s)_2$, set $f((q_s)_1)=(q_s)_2$, mark $(q_s)_2)$ as used

--Define the recursive backtracking function, $dive(v)$, on a vertex $v$

--$dive(v)$ does:

----checks if last vertex is mapped, if that happens, verify isomorphism conditions on $f$, returning True if passes all

----if not, if $v$ already mapped then move to next, until an unmapped vertex is found

----using a loop, try every unused state $x$ of $Q_2$ as image of $v$ under $f$, verifing the isomorphism conditions hold, if violated, break out and try the next $x$ 

----do these two lines: 1. 'if $dive(v+1)$: return True', and after that 2. reverse $f(v)$ to $-1$ and set $x$ back to unusued (this line will only be done if False is returned)

----these two lines are crucial, since $dive(v+1)$ will eventually reach line 1, which procedes with $dive(v+2)$ and so on. Finally $dive(n)$ will be reached, this is the 'every(last) vertex is mapped' case, now dive verifies the isomorphism conditions and either return a True or a False

----if True is returned, the line if $dive(n-1)$: will return True, then the line if $dive(n-2)$ will also, ..., finally $dive(v+1)$ returns True

----if False is returned, this is a dead end, all possible constructions failed, hence does line 2 (which can be interpreted as going one step back, this might succeed or have to go further back.

----lastly $dive(v)$ returns False. This will only be triggered if all possible constructions from $v$ has failed ($v+1, v+2, ..., n$ has all returned False after trying every possible path), so need to move back.

Now out of dive

--let is_strict_isomorphism return $dive(0)$

--this will either fully construct $f$ and return a True, or will exhaust all possible constuctions from $0$ and finally having $dive(0)$ returning a False

**Discussion about exsistance of a bijection**

Since being strictly isomorphic means 'there exists an isomorphism between the DFAs' and we constucted a specific one. If the $f$ constructed fails to be an isomorphism, could there still exist an isomorphism $g$, making the DFAs isomorphic? 

From how backtracking search works, is_strict_isomorphism searches *every* bijection that maps $(q_s)_1$ to $(q_s)_2$. It tries every possible mapping to an unmapped state, so if an isomorphism existis, it *will* find one. So returning a False really means there does not exist such an isomorphism

In [2]:
def is_strict_isomorphism(A1,A2):
    """
    Tests if two DFAs A1=(Q1,Σ,τ1,qs1,F1) and A2=(Q2,Σ,τ2,qs2,F2) are strictly isomorphic.
    Assumes they have the common alphabet Σ.
    Assumes every state in Q is reachable from qs.
    Uses backtrack search.

    Parameters:
    - A1, A2: Two DFAs
    Arguments of A1, A2:
    - Q1, Q2: some finite set of states
    - Σ: some finite alphabet (a set)
    - τ1, τ2: transition function τi:Qi×Σ→Qi for i=1,2 (a dict)
    - qs1,qs2: start state
    - F1, F2: set of accepting or final states

    Returns: 
    Boolean True or False
    """
    # Store the arguments of the DFAs in variables(if not already in this form)
    Q1,Σ,τ1,qs1,F1 = A1
    Q2,Σ,τ2,qs2,F2 = A2
    # Test the cardinalities of the Qs and the Fs are equal
    if len(Q1)!=len(Q2):
        return False
    if len(F1) != len(F2):
        return False
    # Build the mapping, since Q1 = Q2 = {0,1,...,n-1}, injective implies bijective.
    n = len(Q1)
    f = [-1]*n   # for q in Q1, f[q]=f(q), the image of q in Q2, -1 means not mapped yet.
    used = [False] * n   # Track which states of Q2 are already used, ensures f injective, hence bijective
    # Map qs1 to qs2 i.e. f(qs1)=qs2
    f[qs1] = qs2
    used[qs2] = True
    # Define the recursive backtracking function
    def dive(v):   # v is a vertex/state, will run dive from v=0
        if v == n:   # If we have mapped all vertices, verify the conditions are satisfied
            for q in Q1:
                if (q in F1)!=(f[q] in F2):
                    return False
                for a in Σ:
                    if τ2[(f[q], a)] != f[τ1[(q, a)]]:
                        return False
            return True   # f satisfies the conditions and is a bijection, i.e. isomorphism, so A1 A2 stricly isomorphic.
        if f[v] != -1:
            return dive(v+1)     # If the current vertex is already mapped, skip to the next
        # Try every unused state in Q2, x, as a image for v (i.e. to see if we can set f(v)=x)
        for x in Q2:
            if used[x]:   
                continue   # x has been used, get a new x
            # Check f(F1)=F2, if false can jump right into next loop, saving time 
            if (v in F1) != (x in F2):
                continue
            # Check consistency with already mapped vertices u in Q1
            consistent = True
            for u in Q1:
                if f[u] == -1:
                    continue   # Find those mapped vertices, skipping the unmapped
                for a in Σ:
                    if τ1[(u,a)] == v:   # If from u we go to v on some symbol,
                        if τ2[(f[u],a)] != x:
                            consistent = False   # x is not f(v), since f(v)=f(τ1(u,a))=τ2(f(u),a)
                            break
                    if τ1[(v,a)] == u:   # If from v we go to u on some symbol,
                        if τ2[(x,a)] != f[u]:   
                            consistent = False   # if x=f(v), then f(u)=f(τ1(v,a))=τ2(f(v),a)=τ2(x,a)
                            break
                if not consistent:   # If consistent is False, break out immediately, if True, loop on with next u.
                    break
            if not consistent:   # If consistent is False, this x is inconsistent, go to next x
                continue
            # If reaches here, consistent is True throughout, so try to map v to x, then run dive reapeatedly.
            f[v] = x
            used[x] = True
            if dive(v+1):
                return True   # To reach here, must reached v=n and f[v:n] satisfies the isomorphism conditions.
            f[v]=-1   # Reaching here implies that something went wrong in the if dive(v+1), dive(v+2),...loops
            used[x] = False    # so this x does not work try with another x and all the succeeding vs and xs are reverted
        # Reaching here means that no x worked, so v cant have a image satisfying the isomorphism conditions.
        return False   # This will show that v is a 'dead end' after every possible extenstion, so having to go to higher up (v-1)
    return dive(0)   # This will either fully construct f and returning a True, or,  after trying every possible path, return to dive(0) and return a False

## **2. (d) (Strict)**

Strict isomorphisms will fully preserve synchronization properties of DFAs (if at all). This is since two strict isomorphic DFAs can be thought as renaming the states, with structures identical. 

To explicitly show this, let $A_1=(Q_1,Σ,τ_1,(q_s)_1,F_1)$ and $A_2=(Q_2,Σ,τ_2,(q_s)_2,F_2)$ be strictly isomorphic, $f$ be an isomorphism between them. For $x∈Q_1$ define $t_1(x,w)=y$ to be transition function of words, so using a word $w$ on a state $x$ ends at state $y$. With $w=w_1w_2...w_n$ where each $w_i∈Σ$ for finite natural $i$, $t(x,w)$ is basically $τ_1(...(τ_1(τ_1(x,w_1),w_2),...,),w_n)$. $t_2(f(q),w)=f(t_1(q,w))$ for all $q∈Q_1$ and all word $w$ over $Σ$ can be shown to hold using induction, but too length to write here.

If $A_1$ is synchronizing, there exists a reset word $w$ over $Σ$, and resets to a state $z∈Q_1$, i.e. $t_1(q,w)=z$ for all $q∈Q_1$. Then $f(z)=f(t_1(q,w))=t_2(f(q),w)$ for all $f(q)=q_2∈Q_2$ (by surjectivity of $f$), so $w$ is a reset word and $f(z)∈Q_2$ is the state that is resetted to, so $A_2$ is synchronizing

## Below are functions to genereate DFAs and isomorpic/non-isomorphic DFAs by made by deepseek

Used to test 'is_strict_isomorphism'

In [4]:
import random

def generate_dfa(n, m, f_size):
    """
    Generate a random DFA A = (Q, sigma, tau, qs, F) with:
        Q = {0, 1, ..., n-1}
        sigma = {0, 1, ..., m-1}
        tau: dict with keys (state, symbol) -> next state
        qs = 0 (start state)
        F = set of f_size accepting states

    Ensures every state is reachable from the start state.
    """
    if f_size < 0 or f_size > n:
        raise ValueError("f_size must be between 0 and n")

    # Track available symbols per state (to avoid collisions when building the spanning tree)
    available = [set(range(m)) for _ in range(n)]

    # Transition function: dict (state, symbol) -> next_state
    tau = {}

    # Build a spanning tree from state 0 to all other states
    reachable = [0]          # states already in the tree
    remaining = set(range(1, n))   # states not yet added

    while remaining:
        # Pick a random state from reachable that still has free symbols
        candidates = [s for s in reachable if available[s]]
        u = random.choice(candidates)

        # Pick a random unused symbol for u
        a = random.choice(list(available[u]))

        # Pick a random target from the remaining states
        v = random.choice(list(remaining))

        # Assign the transition
        tau[(u, a)] = v
        available[u].remove(a)

        # Update sets
        reachable.append(v)
        remaining.remove(v)

    # Fill all remaining transitions arbitrarily
    for u in range(n):
        for a in range(m):
            if (u, a) not in tau:
                tau[(u, a)] = random.randint(0, n-1)

    # Choose accepting states
    F = set(random.sample(range(n), f_size))

    # Q and sigma as sets (as described)
    Q = set(range(n))
    sigma = set(range(m))
    qs = 0

    return Q, sigma, tau, qs, F

In [5]:
def isomorphic_dfa(dfa, perm=None):
    """
    Return a new DFA that is isomorphic to the input DFA by applying a state permutation.

    Parameters
    ----------
    dfa : tuple (Q, sigma, tau, qs, F)
        The original DFA.
    perm : dict, optional
        A dictionary mapping each old state to a new state.
        If None, a random permutation of the state set is generated.

    Returns
    -------
    new_dfa : tuple
        The isomorphic DFA with states relabeled according to the permutation.
    """
    Q, sigma, tau, qs, F = dfa
    n = len(Q)

    # Generate a random permutation if none is provided
    if perm is None:
        # Create a random permutation of the states (list where new_state = perm_list[old_state])
        perm_list = list(range(n))
        random.shuffle(perm_list)
        old_to_new = {old: new for old, new in enumerate(perm_list)}
    else:
        old_to_new = perm

    # Apply the permutation to all components
    new_tau = {}
    for (s, a), t in tau.items():
        new_tau[(old_to_new[s], a)] = old_to_new[t]

    new_qs = old_to_new[qs]
    new_F = {old_to_new[f] for f in F}
    new_Q = set(old_to_new.values())          # should equal the original set of integers

    return new_Q, sigma, new_tau, new_qs, new_F

def non_isomorphic_dfa(dfa, state=None, symbol=None, new_target=None):
    """
    Return a new DFA obtained by changing one transition of the input DFA.

    Parameters
    ----------
    dfa : tuple (Q, sigma, tau, qs, F)
        The original DFA.
    state : int, optional
        The source state of the transition to modify. If None, a random state is chosen.
    symbol : int, optional
        The alphabet symbol. If None, a random symbol is chosen.
    new_target : int, optional
        The new target state. If None, a random state from Q is chosen.

    Returns
    -------
    new_dfa : tuple
        The DFA with the specified transition replaced.
    """
    Q, sigma, tau, qs, F = dfa
    n = len(Q)

    # Choose random values if not specified
    if state is None:
        state = random.choice(list(Q))
    if symbol is None:
        symbol = random.choice(list(sigma))
    if new_target is None:
        new_target = random.choice(list(Q))

    # Create a modified copy of the transition dictionary
    new_tau = tau.copy()
    new_tau[(state, symbol)] = new_target

    return Q, sigma, new_tau, qs, F

In [13]:
for i in range(25):
    A1=generate_dfa(10,4,7)
    A2=isomorphic_dfa(A1)
    A3=non_isomorphic_dfa(A1)
    print(is_strict_isomorphism(A1,A2),is_strict_isomorphism(A1,A3))

True False
True False
True False
True False
True False
True False
True True
True False
True False
True False
True False
True False
True False
True False
True False
True False
True False
True False
True False
True True
True False
True False
True False
True False
True False


### **Below is code using wrong search method, ignore it. Kept incase needed**

In [5]:
from collections import deque
def is_strict_isomorphism_BFS(A1,A2):
    """
    Tests if two DFAs A1=(Q1,Σ,τ1,qs1,F1) and A2=(Q2,Σ,τ2,qs2,F2) are stricly isomorphic.
    Assumes they have the common alphabet Σ.
    Assumes every state in Q is reachable from qs.

    Parameters:
    - A1, A2: Two DFAs
    Arguments of A1, A2:
    - Q1, Q2: some finite set of states
    - Σ: some finite alphabet (a set)
    - τ1, τ2: transition function τi:Qi×Σ→Qi for i=1,2 (a dict)
    - qs1,qs2: start state
    - F1, F2: set of accepting or final states

    Returns: 
    Boolean True or False
    """
    # Store the arguments of the DFAs in variables(if not already in this form)
    Q1,Σ,τ1,qs1,F1=A1
    Q2,Σ,τ2,qs2,F2=A2
    # Test the cardinalities of the Qs and the Fs are equal
    if len(Q1)!=len(Q2):
        return False
    if len(F1)!=len(F2):
        return False
    # Build the mapping, since assmued the Qs are {0,1,...,n-1}, the mapping will turn out to be a permutation.
    f=[-1]*len(Q1)   # Note f initially has -1 in every entry, not 0, as there's a vertex called 0
    # Map qs1 to qs2 i.e. f(qs1)=qs2
    f[qs1]=qs2
    # Now construct the rest of f using Breadth-first search (BFS), first make a queue containg the tuple (qs1,qs2)        
    queue=deque()
    queue.append((qs1,qs2))
    # While there are still things in the queue
    while queue:
        # Get the first thing of the queue, note q1 is in Q1 and q2 is in Q2
        q1,q2=queue.popleft()
        # Run through every number in the alphabet
        for a in Σ:
            # Use τ to get the next states, called ns
            ns1=τ1[(q1,a)]
            ns2=τ2[(q2,a)]
            # If ns1 is unmapped, map it to ns2 and throw in to the queue
            if f[ns1]==-1:
                f[ns1]=ns2
                queue.append((ns1,ns2))
    # BFS is done, check f is indeed a bijection from Q1 to Q2
    if len(set(f))!=len(f):  # Check injectivity (injective if len(set(f))==len(f)) 
        # This is as f(i)=f[i] for i=0,1,...,n-1, if f(x)=f(y) for some x≠y, then len(set(f))<len(f)=n,
        # Using that Q1=Q2={0,1,...,n-1}, and a theorem that injections from a set to itself is surjective,
        # We can conclude if len(set(f))==len(f), then f is indeed a bijection.
        return False 
    # Now check if f(F1)=F2, by checking every q in Q1 that is in F1 has f(q) in F2, and vice versa
    for q in Q1:
        if (q in F1)!=(f[q] in F2):
            return False
        # Check if τ2(f(q),a)=f(τ1(q,a)) for all q∈Q1 and all a∈Σ
        for a in Σ:
            if τ2[(f[q],a)]!=f[τ1[(q,a)]]:
                return False
    # If reaches here, then satisfies all the reqirements so f is an isomorphism and A1 strictly isomorphic to A2
    return True